In [1]:
"""
Build assumption_master sheet: fc_db_loading
-----
FC DB loading — multiplier for members with Status='DB'. 5-row step function.
Source: Basis and Assumptions.xlsx > Assumptions > E418:F422.
Anchor label: 'Months of Disablement' at Assumptions!E417.

FC row-4 formula wiring (V1.7.43):
  BA (DB Loading) = INDEX(Assumptions!$E$418:$F$422,
                          MIN(5, FLOOR((Comm_Date - F4)/365.25, 1) + 1), 2)
Positional index (MIN cap 5) → years_disabled 0..4.

Author: Mohamed Hussein Gadiwala
Assumption version: A2026_07 (assumption master)
"""

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import csv
import openpyxl
import polars as pl

BASE = Path(r"J:\Product & Client Solutions\Toolbox\Calculators"
            r"\Business Tools\Tender Summary\GAP Pricing Engine")

SOURCE_XLSX  = BASE / "Basis and Assumptions.xlsx"
SOURCE_SHEET = "Assumptions"

ASSUMPTION_VERSION = "A2026_07"
SNAPSHOT_DIR = BASE / "assumption_snapshots" / f"assumption_{ASSUMPTION_VERSION}"
MASTER_XLSX  = BASE / "assumption_master" / f"assumption_master_{ASSUMPTION_VERSION}.xlsx"
BUILD_LOG    = SNAPSHOT_DIR / "build_log.csv"

SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)
MASTER_XLSX.parent.mkdir(parents=True, exist_ok=True)

SHEET_NAME   = "fc_db_loading"
PARQUET_PATH = SNAPSHOT_DIR / f"{SHEET_NAME}.parquet"

BUILT_AT = datetime.now(timezone.utc).isoformat(timespec="seconds")
BUILT_BY = "Mohamed Hussein Gadiwala"

wb = openpyxl.load_workbook(SOURCE_XLSX, data_only=True)
ws = wb[SOURCE_SHEET]
SOURCE_SHA = hashlib.sha256(SOURCE_XLSX.read_bytes()).hexdigest()

print(f"Building: {SHEET_NAME}")
print(f"Source:   {SOURCE_XLSX.name} > {SOURCE_SHEET}")
print(f"Built at: {BUILT_AT}")


Building: fc_db_loading
Source:   Basis and Assumptions.xlsx > Assumptions
Built at: 2026-07-22T10:54:51+00:00


In [2]:
def find_all_cells_any_col(ws, label: str, max_row: int = 1200, max_col: int = 30) -> list[tuple[int, int]]:
    """Find ALL cells matching `label` exactly, anywhere on the sheet."""
    hits = []
    for row in range(1, max_row + 1):
        for col in range(1, max_col + 1):
            v = ws.cell(row=row, column=col).value
            if isinstance(v, str) and v.strip() == label:
                hits.append((row, col))
    return hits


def find_in_range(ws, label: str, row_start: int, row_end: int, col: int | None = None) -> tuple[int, int] | None:
    """Find first cell matching `label` within given row range (and optional col)."""
    hits = find_all_cells_any_col(ws, label)
    for r, c in hits:
        if row_start <= r < row_end and (col is None or c == col):
            return (r, c)
    return None


def get_col_letter(col: int) -> str:
    return openpyxl.utils.get_column_letter(col)


In [3]:
ANCHOR_LABEL = "Months of Disablement"   # ← was "Months of Disability"
hit = find_in_range(ws, ANCHOR_LABEL, 415, 425, col=5)
assert hit is not None, f"Anchor {ANCHOR_LABEL!r} not found"
anchor_row, anchor_col = hit
print(f"Anchor at Assumptions!{get_col_letter(anchor_col)}{anchor_row}")

DATA_ROW_START = anchor_row + 1
DATA_ROW_END   = anchor_row + 6

EXPECTED_LABELS = ["0 to 12", "12 to 24", "24 to 36", "36 to 48", "> 48"]

all_rows: list[dict] = []
for i, excel_row in enumerate(range(DATA_ROW_START, DATA_ROW_END)):
    source_label   = ws.cell(row=excel_row, column=anchor_col).value
    loading_factor = ws.cell(row=excel_row, column=anchor_col + 1).value

    assert isinstance(source_label, str)
    assert isinstance(loading_factor, (int, float))
    label_norm = source_label.strip()
    assert label_norm == EXPECTED_LABELS[i], (
        f"Row {excel_row}: label drift, got {label_norm!r}, expected {EXPECTED_LABELS[i]!r}"
    )

    all_rows.append({
        "product":        "fc",
        "years_disabled": i,
        "loading_factor": float(loading_factor),
        "source_label":   label_norm,
        "source_row":     excel_row,
        "notes":          "FC DB loading applied when Status='DB'. Positional; row 4 = 4+ years.",
    })

print(f"Total rows: {len(all_rows)}")
for r in all_rows:
    print(f"  years_disabled={r['years_disabled']}  loading={r['loading_factor']:.2f}  "
          f"({r['source_label']!r})")

Anchor at Assumptions!E417
Total rows: 5
  years_disabled=0  loading=14.50  ('0 to 12')
  years_disabled=1  loading=9.00  ('12 to 24')
  years_disabled=2  loading=8.00  ('24 to 36')
  years_disabled=3  loading=3.50  ('36 to 48')
  years_disabled=4  loading=1.50  ('> 48')


In [4]:
df = pl.DataFrame(all_rows).select([
    "product", "years_disabled", "loading_factor",
    "source_label", "source_row", "notes",
])

dupes = df.group_by(["product", "years_disabled"]).len().filter(pl.col("len") > 1)
assert dupes.height == 0
assert set(df["product"].unique().to_list()) == {"fc"}

years = sorted(df["years_disabled"].to_list())
assert years == [0, 1, 2, 3, 4]

factors = df.sort("years_disabled")["loading_factor"].to_list()
for i in range(1, len(factors)):
    assert factors[i] <= factors[i-1]

def get_loading(years_disabled):
    return float(df.filter(pl.col("years_disabled") == years_disabled)["loading_factor"][0])

assert abs(get_loading(0) - 14.5) < 1e-6
assert abs(get_loading(1) -  9.0) < 1e-6
assert abs(get_loading(2) -  8.0) < 1e-6
assert abs(get_loading(3) -  3.5) < 1e-6
assert abs(get_loading(4) -  1.5) < 1e-6

print(f"✓ {df.height} rows validated")
print(f"✓ monotonically non-increasing: {factors}")
print(f"✓ 5 frozen checks passed")
print(df)


✓ 5 rows validated
✓ monotonically non-increasing: [14.5, 9.0, 8.0, 3.5, 1.5]
✓ 5 frozen checks passed
shape: (5, 6)
┌─────────┬────────────────┬────────────────┬──────────────┬────────────┬───────────────────────┐
│ product ┆ years_disabled ┆ loading_factor ┆ source_label ┆ source_row ┆ notes                 │
│ ---     ┆ ---            ┆ ---            ┆ ---          ┆ ---        ┆ ---                   │
│ str     ┆ i64            ┆ f64            ┆ str          ┆ i64        ┆ str                   │
╞═════════╪════════════════╪════════════════╪══════════════╪════════════╪═══════════════════════╡
│ fc      ┆ 0              ┆ 14.5           ┆ 0 to 12      ┆ 418        ┆ FC DB loading applied │
│         ┆                ┆                ┆              ┆            ┆ when Sta…             │
│ fc      ┆ 1              ┆ 9.0            ┆ 12 to 24     ┆ 419        ┆ FC DB loading applied │
│         ┆                ┆                ┆              ┆            ┆ when Sta…             │
│

In [5]:
df.write_parquet(PARQUET_PATH)
parquet_bytes = PARQUET_PATH.read_bytes()
PARQUET_SHA = hashlib.sha256(parquet_bytes).hexdigest()
PARQUET_SIZE_KB = len(parquet_bytes) / 1024

print(f"✓ Wrote {PARQUET_PATH.name} ({PARQUET_SIZE_KB:.2f} KB)")

if MASTER_XLSX.exists():
    master_wb = openpyxl.load_workbook(MASTER_XLSX)
else:
    master_wb = openpyxl.Workbook()
    if "Sheet" in master_wb.sheetnames:
        del master_wb["Sheet"]

if SHEET_NAME in master_wb.sheetnames:
    del master_wb[SHEET_NAME]

ws_out = master_wb.create_sheet(SHEET_NAME)
ws_out.append(df.columns)
for row in df.iter_rows():
    ws_out.append(list(row))

master_wb.save(MASTER_XLSX)
print(f"✓ Master workbook updated")


✓ Wrote fc_db_loading.parquet (2.79 KB)
✓ Master workbook updated


In [6]:
LOG_HEADERS = [
    "built_at", "built_by", "sheet", "rows", "cols",
    "parquet_path", "parquet_size_kb", "parquet_sha256",
    "source_file", "source_sheet", "source_sha256",
    "assumption_version", "notes",
]

log_row = {
    "built_at": BUILT_AT, "built_by": BUILT_BY, "sheet": SHEET_NAME,
    "rows": df.height, "cols": len(df.columns),
    "parquet_path": str(PARQUET_PATH.relative_to(BASE)),
    "parquet_size_kb": f"{PARQUET_SIZE_KB:.2f}", "parquet_sha256": PARQUET_SHA,
    "source_file": SOURCE_XLSX.name, "source_sheet": SOURCE_SHEET,
    "source_sha256": SOURCE_SHA, "assumption_version": ASSUMPTION_VERSION,
    "notes": ("FC DB loading (5-row step function). Positional years_disabled 0..4. "
              "Anchor: Assumptions!E417 'Months of Disability'. Consumer: FC row-4 BA."),
}

write_header = not BUILD_LOG.exists()
with BUILD_LOG.open("a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=LOG_HEADERS)
    if write_header:
        writer.writeheader()
    writer.writerow(log_row)

print(f"✓ build_log.csv updated")
print(f"\n{'='*60}\nBUILD COMPLETE: {SHEET_NAME}\n{'='*60}")


✓ build_log.csv updated

BUILD COMPLETE: fc_db_loading
